# `features.py` — ¿de verdad no hay fuga de datos?

## Qué es esto y por qué merece su propio notebook

Ya sabemos, documentado en [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md), que el pipeline
heredado tiene un bug real de fuga de datos: al calcular el promedio de un jugador, una celda
mal escrita terminaba incluyendo el valor de la semana que se quiere predecir dentro de su
propio promedio — es decir, el modelo "hacía trampa" viendo el futuro sin darse cuenta.

Este notebook responde una pregunta puntual y esencial, distinta de las dos anteriores: **¿el
código nuevo (`weekly/wr/src/features.py`) evita ese mismo error, de verdad, con evidencia fila
por fila — no solo porque está escrito con cuidado?**

## Qué calcula `features.py`

Para cada jugador y cada estadística (ej. yardas de recepción), calcula 3 promedios usando
**solo lo que pasó antes de esa semana**, nunca la semana misma:

- **`_season_avg`**: promedio de la temporada en curso, hasta la semana anterior. Se reinicia en
  cada temporada nueva.
- **`_last{N}_avg`**: promedio de las últimas N apariciones del jugador — sin reiniciar por
  temporada, porque la "forma reciente" de un jugador no se resetea el 1 de septiembre.
- **`_career_avg`**: promedio de toda su carrera hasta la semana anterior.

`target_share`, `air_yards_share` y `wopr` **no se calculan aquí** — ya vienen listos y limpios
desde `cargar_stats_semanales()` (verificado en `02_construccion_datos.ipynb`); el pipeline
heredado los recalculaba a mano desde *play-by-play*, dos veces por notebook, sin necesidad.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features

stats = data.cargar_stats_semanales([2024, 2025])
wr = stats[stats["position"] == "WR"].copy()

resultado = features.agregar_promedios_jugador(wr, ["receiving_yards"], ventana=3, id_col="player_id")
print("filas:", resultado.shape[0])

filas: 4952


## La prueba: un jugador real, semana por semana

Se toma un receptor con trayectoria larga en ambas temporadas (DeVonta Smith) y se revisa, fila
por fila, si el promedio de cada semana usa solo lo anterior.

In [2]:
smith = resultado[resultado["player_display_name"] == "DeVonta Smith"].sort_values(["season", "week"])
cols = ["season", "week", "receiving_yards", "receiving_yards_season_avg",
        "receiving_yards_last3_avg", "receiving_yards_career_avg"]
smith[cols].head(9)

,season,week,receiving_yards,receiving_yards_season_avg,receiving_yards_last3_avg,receiving_yards_career_avg
595,2024,1,84,NaN,NaN,NaN
1639,2024,2,76,84.000000,84.000000,84.000000
2685,2024,3,79,80.000000,80.000000,80.000000
5659,2024,6,64,79.666667,79.666667,79.666667
6656,2024,7,-2,75.750000,73.000000,75.750000
7697,2024,8,85,60.200000,47.000000,60.200000
8716,2024,9,87,64.333333,49.000000,64.333333
9675,2024,10,14,67.571429,56.666667,67.571429
10585,2024,11,29,60.875000,62.000000,60.875000


**Cómo se verifica que no hay fuga, con números concretos:**

- **Semana 1 de 2024**: todos los promedios salen `NaN`. Tiene sentido — no existe ninguna
  semana anterior de la cual promediar. Si aquí saliera un número, sería la primera señal de que
  algo se está calculando mal.
- **Semana 2**: `season_avg` = 84.0, que es exactamente el valor que tuvo en la semana 1 — no un
  promedio que ya incluya la semana 2 misma.
- **Semana 7** (yardas = -2 esa semana): `season_avg` = 75.75 = promedio de las semanas 1-4
  (84, 76, 79, 64) — **no incluye el -2 de esta misma semana**. `last3_avg` = 73.0 = promedio de
  las 3 semanas *anteriores* (76, 79, 64), tampoco el -2 de hoy.
- **Semana 8**: `last3_avg` = 47.0 = promedio de (79, 64, **-2**) — aquí sí ya entra el -2, pero
  porque ahora es una semana *pasada* respecto a la semana 8, no la semana actual. Así se
  supone que debe moverse una ventana de "los últimos 3 juegos".

Cada uno de estos números se puede recalcular a mano con una calculadora — no es una afirmación,
es aritmética verificable.

## La segunda prueba: ¿`season_avg` se reinicia, y `career_avg` no?

Al cruzar de 2024 a 2025, `season_avg` debería "olvidar" 2024 y empezar de cero, mientras
`career_avg` debería seguir sumando desde donde iba.

In [3]:
smith[cols].tail(5)

,season,week,receiving_yards,receiving_yards_season_avg,receiving_yards_last3_avg,receiving_yards_career_avg
31844,2025,14,37,66.833333,48.333333,65.400000
32838,2025,15,50,64.538462,58.000000,64.307692
33905,2025,16,42,63.500000,45.000000,63.777778
34993,2025,17,25,62.066667,43.000000,63.000000
36038,2025,18,52,59.750000,39.000000,61.689655


En la última fila (semana 18 de 2025), `season_avg` (59.75) y `career_avg` (61.69) ya son
números distintos — si `season_avg` no se hubiera reiniciado, serían iguales o muy parecidos
todo el tiempo. Esa diferencia es la prueba de que el corte por temporada funciona como se
espera.

## Por qué el código no puede repetir el bug viejo

El bug documentado en `HALLAZGOS.md` viene de usar `groupby().apply()` con una función que
regresa una tabla nueva, y luego pegarla de vuelta sin `reset_index` (o, en pandas nuevo, con un
índice de tipo distinto al esperado) — el pegado queda desalineado y cada jugador termina con el
promedio de otro.

`agregar_promedios_jugador()` usa `groupby(...).transform(...)` en vez de `.apply(...)` — con
`transform`, pandas garantiza que el resultado se alinea de vuelta al índice original de cada
fila automáticamente, sin ningún paso manual que se pueda hacer mal. No es solo "se escribió con
cuidado" — es una forma en la que ese tipo de error no tiene dónde ocurrir.

## Qué falta

- Decidir con evidencia (Fase 3) si `last3` o `last5` predice mejor — hoy `ventana` es un
  parámetro, no una decisión tomada.
- Agregados a nivel equipo (ofensiva/defensiva propia) quedan para la Fase 4, si la evidencia
  muestra que valen la pena — no se agregan por costumbre.

Código fuente: [`../src/features.py`](../src/features.py) · Notebook anterior:
[`02_construccion_datos.ipynb`](02_construccion_datos.ipynb)